# PDF Analyzer — servidor en Google Colab

Arquitectura: **tu navegador (index.html local)** → túnel HTTPS → **Flask :5050 (Colab)** → **Ollama :11434 (Colab)**

1. `Entorno de ejecución → Cambiar tipo → GPU T4`
2. Ejecuta las celdas en orden.
3. La última celda imprime la URL para abrir `interface/index.html?api=...` en tu PC.

> El túnel cambia de URL cada vez que reinicias la sesión.

In [ ]:
# ── 1. Parámetros ────────────────────────────────────────────────
REPO       = 'https://github.com/gzjhones/chatbot-pdf-qwen.git'
MODEL_NAME = 'qwen3:1.7b'
PORT       = 5050
API_KEY    = 'cambia-esta-clave'   # '' para desactivar

In [ ]:
# ── 2. Código del proyecto ──────────────────────────────────────
%cd /content
!rm -rf chatbot-pdf-qwen && git clone -q $REPO
%cd /content/chatbot-pdf-qwen
!pip -q install flask flask-cors pymupdf requests

In [ ]:
# ── 3. Instalar y arrancar Ollama ───────────────────────────────
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests, os
ollama_proc = subprocess.Popen(['ollama', 'serve'],
                               stdout=open('/content/ollama.log', 'w'),
                               stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get('http://localhost:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
print('Ollama OK')
!ollama pull $MODEL_NAME
!ollama list

In [ ]:
# ── 4. Arrancar Flask en segundo plano ──────────────────────────
env = dict(os.environ, MODEL_NAME=MODEL_NAME, PORT=str(PORT), API_KEY=API_KEY,
           OLLAMA_URL='http://localhost:11434')
flask_proc = subprocess.Popen(['python', 'server/app.py'], env=env,
                              stdout=open('/content/flask.log', 'w'),
                              stderr=subprocess.STDOUT)
time.sleep(3)
print(requests.get(f'http://localhost:{PORT}/health',
                   headers={'X-API-Key': API_KEY}).json())

In [ ]:
# ── 5. Túnel público con Cloudflare (sin cuenta) ────────────────
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

import re
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}'],
    stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        PUBLIC_URL = m.group(0); break

print('URL pública :', PUBLIC_URL)
print('Abre en tu PC:')
print(f'  interface/index.html?api={PUBLIC_URL}' + (f'&key={API_KEY}' if API_KEY else ''))

In [ ]:
# ── 6. Prueba desde Colab a través del túnel ───────────────────
time.sleep(5)  # el DNS del túnel tarda unos segundos
print(requests.get(PUBLIC_URL + '/health', headers={'X-API-Key': API_KEY}, timeout=30).json())

### Alternativa: ngrok
Si prefieres un dominio fijo, crea cuenta en ngrok, copia tu authtoken y reemplaza la celda 5 por:
```python
!pip -q install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token('TU_TOKEN')
PUBLIC_URL = ngrok.connect(PORT).public_url
print(PUBLIC_URL)
```
El widget ya envía la cabecera `ngrok-skip-browser-warning`, así que no aparece la página de aviso.

### Logs
`!tail -n 30 /content/flask.log` · `!tail -n 30 /content/ollama.log` · `!tail -n 30 /content/tunnel.log`